# Preprocessing step by step 

This notebook presents step-by-step preprocessing steps to get to know the dataset and evaluate its features

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### 1. Load the dataset 
Load the data and print some of the attributes

In [4]:
df = pd.read_csv('../dataset/training_data.csv')

print(df.head())

                                               title                date  \
0  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
1  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
2  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
3  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
4  The President's News Conference in Hanoi, Vietnam  September 10, 2023   

         president                                                url  \
0  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
1  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
2  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
3  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
4  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   

   question_order                                 interview_question  \
0               1  Q. Of the Bid

In [5]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3448 entries, 0 to 3447
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  3448 non-null   object 
 1   date                   3448 non-null   object 
 2   president              3448 non-null   object 
 3   url                    3448 non-null   object 
 4   question_order         3448 non-null   int64  
 5   interview_question     3448 non-null   object 
 6   interview_answer       3448 non-null   object 
 7   gpt3.5_summary         3448 non-null   object 
 8   gpt3.5_prediction      3448 non-null   object 
 9   question               3448 non-null   object 
 10  annotator_id           3448 non-null   int64  
 11  annotator1             0 non-null      float64
 12  annotator2             0 non-null      float64
 13  annotator3             0 non-null      float64
 14  inaudible              3448 non-null   bool   
 15  mult

In [6]:
print(df.describe(include="all"))

                                  title               date        president  \
count                              3448               3448             3448   
unique                              175                287                4   
top     The President's News Conference  November 07, 2018  Donald J. Trump   
freq                               1626                117             1325   
mean                                NaN                NaN              NaN   
std                                 NaN                NaN              NaN   
min                                 NaN                NaN              NaN   
25%                                 NaN                NaN              NaN   
50%                                 NaN                NaN              NaN   
75%                                 NaN                NaN              NaN   
max                                 NaN                NaN              NaN   

                                                   

Check for missing values

In [7]:
print(df.isna().sum())

title                       0
date                        0
president                   0
url                         0
question_order              0
interview_question          0
interview_answer            0
gpt3.5_summary              0
gpt3.5_prediction           0
question                    0
annotator_id                0
annotator1               3448
annotator2               3448
annotator3               3448
inaudible                   0
multiple_questions          0
affirmative_questions       0
index                       0
clarity_label               0
evasion_label               0
dtype: int64


### 2. Data cleanup

Start cleaning the dataset by removing unnecessary features

Remove null-fetaures and urls.

In [8]:

for i in [1, 2, 3]:
    annotator = 'annotator' + str(i)
    df = df.drop(annotator, axis=1)

df = df.drop('url', axis = 1)



### 3. Q&A mathing 

Distinguish and separate multiple part questions in to separate rows

In [9]:
import re

Start by defining helper functions to separate questions and answers

In [21]:
def gpt_summary_parser(summary):
    """
    The function splits the text in the 'gpt3.5_summary' atrtibute into a question list and an answer list
    
    Args:
        summary (str): the gpt3.5_summary attribute of the dataset

    Returns:
        question_list, answer_list (list): list of questions and answers from the summary (gpt3.5 summary)

    """

    if not isinstance(summary, str):
        return [],[]
    
    parts = re.split(r'The response provides[^:]*:\s*', summary, maxsplit=1, flags=re.IGNORECASE)
    
    if len(parts) < 2:
        return [],[]
    
    q_block, rest = parts
    a_block = re.split(f'\bOverall\b', rest, maxsplit=1, flags=re.IGNORECASE)[0]

    q_list = [m.strip() for m in re.findall(r'\d+\.\s*(.+)', q_block)]
    a_list = [m.strip() for m in re.findall(r'\d+\.\s*(.+)', a_block)]

    return q_list, a_list


In [35]:
import re

def gpt_summary_parser(summary):
    """
    Split gpt3.5_summary into question and answer lists.
    """
    if not isinstance(summary, str):
        return [], []
    
    # Split questions vs. answers
    parts = re.split(
        r'The response provides[^:]*:\s*',
        summary,
        maxsplit=1,
        flags=re.IGNORECASE
    )
    if len(parts) < 2:
        return [], []
    
    q_block, rest = parts
    # Cut off the "Overall ..." section if present
    a_block = re.split(
        r'\bOverall\b',
        rest,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]

    # IMPORTANT: only match numbered items at the *start of a line*
    q_list = [
        m.strip()
        for m in re.findall(r'^\s*\d+\.\s*(.+)', q_block, flags=re.MULTILINE)
    ]
    a_list = [
        m.strip()
        for m in re.findall(r'^\s*\d+\.\s*(.+)', a_block, flags=re.MULTILINE)
    ]

    return q_list, a_list

In [36]:

def extract_single_answer(summary: str) -> str | None:
    """
    For summaries where there's effectively only one question,
    return everything starting from 'The response...' onward.
    """
    if not isinstance(summary, str):
        return None
    
    s_lower = summary.lower()
    
    # Best case: there's a 'The response ...' phrase
    idx = s_lower.find("the response")
    if idx != -1:
        return summary[idx:].strip()
    
    # Fallback: if 'The question consists' exists, take text after it
    idx2 = s_lower.find("the question consists")
    if idx2 != -1:
        parts = summary[idx2:].split("\n", 1)
        if len(parts) == 2:
            return parts[1].strip()
    
    # Last resort: whole summary
    return summary.strip()

Use the helper function to split the summary in to Q/A pairs and add new attributes to the dataset. 

In [37]:
df['answer'] = None

for summary, group in df.groupby('gpt3.5_summary'):
    q_list, a_list = gpt_summary_parser(summary)

    # --- NEW: if parser failed but this summary corresponds to a single question,
    #          just grab the whole "The response..." block as the answer ---
    # (You can also use another criterion here if you prefer, e.g. question_order, etc.)
    unique_questions = group['question'].dropna().unique()
    if (not q_list or not a_list) and len(unique_questions) == 1:
        single_ans = extract_single_answer(summary)
        df.loc[group.index, 'answer'] = single_ans
        continue

    # If we *do* have structured q/a lists, keep your original logic
    if not q_list or not a_list:
        continue

    mapping = {
        q.strip(): a.strip() for q, a in zip(q_list, a_list)
    }

    for idx, row in group.iterrows():
        q_text = str(row['question']).strip()
        subans = mapping.get(q_text)
        df.at[idx, 'answer'] = subans

In [40]:

df[["interview_question", "question", "answer"]].head(20)

,interview_question,question,answer
0,Q. Of the Biden administration. And accused th...,How would you respond to the accusation that t...,The President expresses sincerity about gettin...
1,Q. Of the Biden administration. And accused th...,Do you think President Xi is being sincere abo...,China is changing some of the rules of the gam...
2,Q. No worries. Do you believe the country's sl...,Do you believe the country's slowdown and gro...,None
3,Q. No worries. Do you believe the country's sl...,Are you worried about the meeting between Pre...,None
4,"Q. I can imagine. It is evening, I'd like to r...",Is the President's engagement with Asian coun...,The President mentions the deals and pacts sig...
5,"Q. I can imagine. It is evening, I'd like to r...",Is there a danger of a cold war?,The President emphasizes that his approach is ...
6,"Q. I can imagine. It is evening, I'd like to r...",When will the President meet Mr. Xi?,The President expresses his hope to meet Mr. X...
7,Q. It's Aurelia End for AFP. I had a question ...,How concerned are you about this lack of cons...,The response provides the following informatio...
8,"Q. Well, let me ask you about—you've spent lot...",Concerns about the lack of communication betw...,The interviewee states that although they have...
9,"Q. Well, let me ask you about—you've spent lot...",Inquiry about the reaction of Kyiv regarding ...,The interviewee acknowledges that the issue of...


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3448 entries, 0 to 3447
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   title                  3448 non-null   object
 1   date                   3448 non-null   object
 2   president              3448 non-null   object
 3   question_order         3448 non-null   int64 
 4   interview_question     3448 non-null   object
 5   interview_answer       3448 non-null   object
 6   gpt3.5_summary         3448 non-null   object
 7   gpt3.5_prediction      3448 non-null   object
 8   question               3448 non-null   object
 9   annotator_id           3448 non-null   int64 
 10  inaudible              3448 non-null   bool  
 11  multiple_questions     3448 non-null   bool  
 12  affirmative_questions  3448 non-null   bool  
 13  index                  3448 non-null   int64 
 14  clarity_label          3448 non-null   object
 15  evasion_label        

### 4. Save file 

Save the new dataset to a CSV file

In [12]:
df.to_csv('../dataset/training_data_processed.csv')